# 01 – Nettoyage et vérification (Sujet 9 : Groupe Valmont)

Objectif : passer de 940 lignes brutes à 800 lignes sans doublons, puis à ~794 lignes pour la modélisation, en documentant chaque décision. Ce notebook produit **trois fichiers prêts à l'emploi pour les 9 questions** : chaque membre charge le même fichier, personne ne refait de jointure ni d'encodage.

La section 10 relit les fichiers écrits et contrôle qu'ils sont bien nettoyés : toute anomalie interrompt l'exécution.

**Avant de commiter : Kernel > Restart & Run All**, pour que les sorties soient dans l'ordre.

In [14]:
import pandas as pd
import numpy as np

hotels = pd.read_csv('../data/raw/Valmont_hotels.csv')
reserv = pd.read_csv('../data/raw/Valmont_reservations.csv')
print(hotels.shape, reserv.shape)
print(list(hotels.columns))
reserv.head()

(32, 6) (940, 13)
['hotel_id', 'ville', 'categorie', 'chambres', 'salles_seminaire', 'distance_gare_km']


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule
0,R700357,H45,groupe,1,11.0,15.0,9,non,OTA,80.33,non,9.0,non
1,R700658,H51,loisir,2,4.0,9.0,8,oui,direct,157.02,oui,4.0,non
2,R700649,H41,affaires,5,1.0,7.0,12,oui,direct,434.92,oui,6.0,non
3,R700732,H69,loisir,4,1.0,83.0,2,non,OTA,92.70,non,1.0,oui
4,R700098,H52,loisir,3,6.0,213.0,3,oui,OTA,85.74,oui,8.0,non


## 1. Inspection
Types, manquants et valeurs extrêmes avant de toucher à quoi que ce soit.

In [15]:
print(reserv.dtypes)
print(reserv.isna().sum())
reserv.describe().round(2)

reservation_id              object
hotel_id                    object
segment_declare             object
nb_nuits                     int64
nb_personnes               float64
delai_reservation_j        float64
mois_sejour                  int64
weekend                     object
canal                       object
prix_nuit_eur              float64
petit_dejeuner              object
nuitees_cumulees_client    float64
a_annule                    object
dtype: object
reservation_id              0
hotel_id                    0
segment_declare             0
nb_nuits                    0
nb_personnes               24
delai_reservation_j         0
mois_sejour                 0
weekend                     0
canal                       0
prix_nuit_eur               0
petit_dejeuner             24
nuitees_cumulees_client     0
a_annule                    0
dtype: int64


,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,prix_nuit_eur,nuitees_cumulees_client
count,940.00,916.00,940.00,940.00,940.00,940.00
mean,4.10,3.41,42.31,6.48,193.58,11.23
std,20.48,4.05,40.55,3.47,105.17,8.27
min,1.00,1.00,0.00,1.00,0.00,1.00
25%,2.00,1.00,10.00,3.00,115.48,5.00
50%,3.00,1.00,28.00,6.00,152.36,9.00
75%,4.00,3.00,64.25,10.00,280.97,15.00
max,365.00,18.00,235.00,12.00,481.09,54.00


## 2. Doublons (940 → 800)
On compte les doublons exacts, puis on vérifie que chaque identifiant répété correspond à une ligne strictement identique (sinon ce serait un autre problème qu'un simple doublon).

In [16]:
n_exacts = reserv.duplicated().sum()
n_id = reserv['reservation_id'].duplicated().sum()
print('doublons exacts :', n_exacts, '| identifiants répétés :', n_id)
assert n_exacts == n_id, "Des identifiants répétés ne sont pas des doublons exacts : à investiguer"

df = reserv.drop_duplicates().reset_index(drop=True)
assert df['reservation_id'].is_unique
print('lignes après dédoublonnage :', len(df))

doublons exacts : 140 | identifiants répétés : 140
lignes après dédoublonnage : 800


## 3. Vérification des modalités et des unités
La consigne insiste sur les colonnes dont l'unité n'est pas celle qu'on croit. On contrôle explicitement chaque variable, et on écrit le résultat dans le README, même si rien n'apparaît.

In [17]:
# Modalités des variables catégorielles (fautes de frappe, casse, espaces)
for col in ['segment_declare', 'canal', 'weekend', 'petit_dejeuner', 'a_annule']:
    print(col, '->', df[col].value_counts(dropna=False).to_dict())

assert set(df['weekend'].dropna()) <= {'oui', 'non'}
assert set(df['a_annule'].dropna()) <= {'oui', 'non'}
assert set(df['petit_dejeuner'].dropna()) <= {'oui', 'non'}
assert df['mois_sejour'].between(1, 12).all(), 'mois hors de 1-12'
assert (df['delai_reservation_j'] >= 0).all(), 'délai négatif'
assert (df['nb_nuits'] >= 1).all()

# Espaces parasites éventuels
for col in df.select_dtypes(include=['object', 'string']).columns:
    assert (df[col].dropna() == df[col].dropna().str.strip()).all(), f'espaces dans {col}'

# Unités : prix = EUR par nuit, délai = jours, nb_nuits = nuits entières
print(df[['prix_nuit_eur', 'delai_reservation_j', 'nb_nuits', 'nb_personnes']].describe().round(1))
print('nb_personnes non entier :', (df['nb_personnes'].dropna() % 1 != 0).sum())
print('nb_nuits non entier     :', (df['nb_nuits'] % 1 != 0).sum())

segment_declare -> {'loisir': 329, 'affaires': 321, 'groupe': 150}
canal -> {'OTA': 339, 'direct': 212, 'entreprise': 157, 'agence': 92}
weekend -> {'non': 548, 'oui': 252}
petit_dejeuner -> {'non': 426, 'oui': 354, nan: 20}
a_annule -> {'non': 591, 'oui': 209}
       prix_nuit_eur  delai_reservation_j  nb_nuits  nb_personnes
count          800.0                800.0     800.0         780.0
mean           194.8                 42.1       4.3           3.4
std            105.8                 40.6      22.2           4.1
min              0.0                  0.0       1.0           1.0
25%            115.7                 10.0       2.0           1.0
50%            152.8                 28.0       3.0           1.0
75%            284.3                 64.0       4.0           3.0
max            481.1                235.0     365.0          18.0
nb_personnes non entier : 0
nb_nuits non entier     : 0


## 4. Cohérence avec la table des hôtels
Chaque réservation doit pointer vers un hôtel connu (clé de jointure pour Q3, Q4, Q6, Q9).

In [18]:
assert hotels['hotel_id'].is_unique
print('hotel_id inconnus :', set(df.hotel_id) - set(hotels.hotel_id))
print('hôtels sans réservation :', len(set(hotels.hotel_id) - set(df.hotel_id)))
print(hotels.categorie.value_counts())
print('Réservations par hôtel (min / médiane / max) :',
      df.groupby('hotel_id').size().agg(['min', 'median', 'max']).tolist())

hotel_id inconnus : set()
hôtels sans réservation : 0
categorie
3 étoiles    13
4 étoiles    11
5 étoiles     8
Name: count, dtype: int64
Réservations par hôtel (min / médiane / max) : [14.0, 25.0, 39.0]


## 5. Valeurs manquantes
Seules `nb_personnes` et `petit_dejeuner` sont concernées.

**Décisions :**
- `nb_personnes` : médiane **par segment déclaré** (groupe ≈ 11, loisir ≈ 2, affaires ≈ 1). Une médiane globale (1) sous-estimerait les groupes. Un indicateur `nb_personnes_impute` permet de tester la sensibilité.
- `petit_dejeuner` : modalité `inconnu` plutôt qu'une valeur inventée.
- Limite à écrire dans le README : l'imputation est faite avant le découpage train-test (effet négligeable ici, une vingtaine de lignes sur 800, mais à mentionner en Q5).

In [19]:
print(df.groupby('segment_declare')['nb_personnes'].median())
df['nb_personnes_impute'] = df['nb_personnes'].isna()
df['nb_personnes'] = df['nb_personnes'].fillna(
    df.groupby('segment_declare')['nb_personnes'].transform('median'))
df['petit_dejeuner'] = df['petit_dejeuner'].fillna('inconnu')
assert df[['nb_personnes', 'petit_dejeuner']].isna().sum().sum() == 0
print('imputés :', df['nb_personnes_impute'].sum())

segment_declare
affaires     1.0
groupe      11.0
loisir       2.0
Name: nb_personnes, dtype: float64
imputés : 20


## 6. Valeurs aberrantes évidentes : séjours ≥ 30 nuits et prix ≤ 10 €
On ne supprime **pas en silence** : on les isole dans un fichier à part (réutilisé en Q9) et on les exclut seulement de la base de modélisation.

**Seuils (plus défendables que `== 365` et `== 0`) :** aucun séjour hôtelier réel ne dépasse un mois, et aucune nuitée ne coûte moins de 10 €. On compare avec les valeurs exactes pour voir si les seuils changent quelque chose.

In [20]:
SEUIL_NUITS = 30
SEUIL_PRIX = 10

m_nuits = df.nb_nuits >= SEUIL_NUITS
m_prix = df.prix_nuit_eur <= SEUIL_PRIX
masque = m_nuits | m_prix

print('nb_nuits >= 30 :', m_nuits.sum(), '| dont exactement 365 :', (df.nb_nuits == 365).sum())
print('prix <= 10 EUR :', m_prix.sum(), '| dont exactement 0 :', (df.prix_nuit_eur == 0).sum())
print('lignes concernées (union) :', masque.sum(), '| base de modélisation :', (~masque).sum())

anomalies = df[masque].copy()
anomalies['motif'] = np.select(
    [m_nuits[masque] & m_prix[masque], m_nuits[masque]],
    ['nuits >= 30 et prix <= 10', 'nuits >= 30'], default='prix <= 10')
anomalies

nb_nuits >= 30 : 3 | dont exactement 365 : 3
prix <= 10 EUR : 3 | dont exactement 0 : 3
lignes concernées (union) : 6 | base de modélisation : 794


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,petit_dejeuner,nuitees_cumulees_client,a_annule,nb_personnes_impute,motif
99,R700355,H45,affaires,3,1.0,20.0,3,oui,OTA,0.00,non,5.0,non,False,prix <= 10
146,R700232,H63,loisir,365,2.0,59.0,3,non,entreprise,346.25,oui,10.0,oui,False,nuits >= 30
537,R700103,H41,affaires,1,1.0,5.0,1,oui,direct,0.00,oui,4.0,non,False,prix <= 10
572,R700450,H64,loisir,365,3.0,40.0,2,non,agence,140.57,oui,15.0,non,False,nuits >= 30
678,R700247,H63,loisir,3,6.0,49.0,5,non,direct,0.00,non,1.0,oui,False,prix <= 10
785,R700391,H60,affaires,365,1.0,5.0,6,oui,OTA,131.91,non,2.0,non,False,nuits >= 30


**Groupes et nombre de personnes.** Les valeurs extrêmes de `nb_personnes` (jusqu'à 18) sont cohérentes avec le segment « groupe » : on les conserve. On vérifie qu'aucune grosse réservation ne se cache hors du segment groupe.

In [21]:
gros = df[(df.nb_personnes >= 8) & (df.segment_declare != 'groupe')]
print('réservations >= 8 personnes hors segment groupe :', len(gros))
print(df.groupby('segment_declare')['nb_personnes'].describe().round(1))

réservations >= 8 personnes hors segment groupe : 0
                 count  mean  std  min  25%   50%   75%   max
segment_declare                                              
affaires         321.0   1.0  0.0  1.0  1.0   1.0   1.0   1.0
groupe           150.0  11.1  3.0  5.0  9.0  11.0  13.0  18.0
loisir           329.0   2.2  1.1  1.0  1.0   2.0   3.0   7.0


## 7. Colonne `nuitees_cumulees_client`
Un cumul client devrait être **supérieur ou égal** au nombre de nuits de la réservation en cours.

**Décision :** on garde la colonne mais on la marque (`cumul_incoherent`). En Q5, on compare le modèle **avec et sans** (risque de fuite d'information : la valeur peut être calculée après coup).

In [22]:
df['cumul_incoherent'] = df['nuitees_cumulees_client'] < df['nb_nuits']
print('lignes incohérentes (800) :', df['cumul_incoherent'].sum())
print(df.groupby('cumul_incoherent')['a_annule'].apply(lambda s: (s == 'oui').mean()).round(3))

lignes incohérentes (800) : 78
cumul_incoherent
False    0.253
True     0.333
Name: a_annule, dtype: float64


## 8. Variables prêtes pour les questions
Construites **une seule fois ici** pour que tout le groupe travaille sur les mêmes définitions.

| Variable | Utilité |
|---|---|
| `a_annule_bin`, `weekend_bin` | cibles et variables binaires (Q1, Q2, Q3, Q5, Q6) |
| `petit_dej_oui`, `petit_dej_inconnu` | encodage du petit-déjeuner (Q5, Q7) |
| `categorie`, `etoiles` | jointure hôtel, version numérique 3/4/5 (Q3, Q4, Q6, Q9) |
| `delai_tranche` | tranches de délai (Q2) |
| `saison` | saison calendaire (Q4) – voir remarque |
| `prix_sejour_eur` | prix × nuits (Q6, Q8) |
| `cumul_incoherent` | test avec/sans (Q5) |

**Remarque saison :** la saison est définie sur le **calendrier**, pas sur les prix, pour éviter un raisonnement circulaire en Q4 (on veut voir où la grille se trompe *par saison*). Si le groupe définit « haute saison » autrement (ex. mois 6, 8 et 12 repérés comme plus chers), à décider et écrire dans le README.

In [23]:
def enrichir(d):
    d = d.merge(hotels, on='hotel_id', how='left', validate='m:1')
    assert d['categorie'].notna().all(), 'réservation sans hôtel'
    d['etoiles'] = d['categorie'].str.extract(r'(\d)')[0].astype(int)
    d['weekend_bin'] = (d['weekend'] == 'oui').astype(int)
    d['a_annule_bin'] = (d['a_annule'] == 'oui').astype(int)
    d['petit_dej_oui'] = (d['petit_dejeuner'] == 'oui').astype(int)
    d['petit_dej_inconnu'] = (d['petit_dejeuner'] == 'inconnu').astype(int)
    d['delai_tranche'] = pd.cut(d['delai_reservation_j'],
                                bins=[-1, 7, 30, 60, 120, np.inf],
                                labels=['0-7 j', '8-30 j', '31-60 j', '61-120 j', '>120 j'])
    d['saison'] = d['mois_sejour'].map(
        {12: 'hiver', 1: 'hiver', 2: 'hiver', 3: 'printemps', 4: 'printemps', 5: 'printemps',
         6: 'ete', 7: 'ete', 8: 'ete', 9: 'automne', 10: 'automne', 11: 'automne'})
    d['prix_sejour_eur'] = d['prix_nuit_eur'] * d['nb_nuits']
    return d

df_enrichi = enrichir(df)
df_enrichi['anomalie_evidente'] = masque.values
modelisation = df_enrichi[~df_enrichi['anomalie_evidente']].drop(columns='anomalie_evidente').copy()
df_800 = df_enrichi.copy()          # garde la colonne anomalie_evidente (utile en Q9)
anomalies_fichier = df_enrichi[df_enrichi['anomalie_evidente']].copy()
anomalies_fichier['motif'] = anomalies['motif'].values

assert len(df_800) == len(df) and len(modelisation) == (~masque).sum()
print(modelisation.shape)
modelisation.head()

(794, 28)


,reservation_id,hotel_id,segment_declare,nb_nuits,nb_personnes,delai_reservation_j,mois_sejour,weekend,canal,prix_nuit_eur,...,salles_seminaire,distance_gare_km,etoiles,weekend_bin,a_annule_bin,petit_dej_oui,petit_dej_inconnu,delai_tranche,saison,prix_sejour_eur
0,R700357,H45,groupe,1,11.0,15.0,9,non,OTA,80.33,...,6,0.79,3,0,0,0,0,8-30 j,automne,80.33
1,R700658,H51,loisir,2,4.0,9.0,8,oui,direct,157.02,...,8,1.06,3,1,0,1,0,8-30 j,ete,314.04
2,R700649,H41,affaires,5,1.0,7.0,12,oui,direct,434.92,...,7,1.05,5,1,0,1,0,0-7 j,hiver,2174.60
3,R700732,H69,loisir,4,1.0,83.0,2,non,OTA,92.70,...,1,0.82,3,0,1,0,0,61-120 j,hiver,370.80
4,R700098,H52,loisir,3,6.0,213.0,3,oui,OTA,85.74,...,4,3.63,3,1,0,1,0,>120 j,printemps,257.22


## 9. Fichiers de sortie et récapitulatif

In [24]:
df_800.to_csv('../data/clean/reservations_propres_800.csv', index=False)
modelisation.to_csv('../data/clean/reservations_modelisation_794.csv', index=False)
anomalies_fichier.to_csv('../data/clean/anomalies_a_part.csv', index=False)

recap = pd.DataFrame({
    'étape': ['Lignes brutes', 'Après suppression des doublons', 'Base de modélisation'],
    'lignes': [len(reserv), len(df), len(modelisation)]})
print(recap)
print("Taux global d'annulation (800) :", round(df_800['a_annule_bin'].mean(), 3))
print("Taux global d'annulation (modélisation) :", round(modelisation['a_annule_bin'].mean(), 3))

                            étape  lignes
0                   Lignes brutes     940
1  Après suppression des doublons     800
2            Base de modélisation     794
Taux global d'annulation (800) : 0.261
Taux global d'annulation (modélisation) : 0.261


## 10. Vérification finale des fichiers produits
On relit les trois fichiers écrits à la section 9 et on contrôle : comptes de lignes, doublons, valeurs manquantes, modalités et plages, jointure avec les hôtels, cohérence des variables créées.

Chaque contrôle affiche `OK` ou `ECHEC`. Si au moins un contrôle échoue, la cellule s'arrête avec une erreur : il faut corriger le nettoyage avant de passer aux questions. Les seuils `SEUIL_NUITS` et `SEUIL_PRIX` sont ceux de la section 6.

In [25]:
F_MODEL = '../data/clean/reservations_modelisation_794.csv'
F_800 = '../data/clean/reservations_propres_800.csv'
F_ANOM = '../data/clean/anomalies_a_part.csv'

resultats = []
def check(nom, condition, detail=''):
    ok = bool(condition)
    resultats.append(ok)
    print(f"[{'OK   ' if ok else 'ECHEC'}] {nom}" + (f'  -> {detail}' if detail and not ok else ''))

mod = pd.read_csv(F_MODEL)
d800 = pd.read_csv(F_800)
anom = pd.read_csv(F_ANOM)

print('--- 1. Comptes de lignes ---')
n_uniques = reserv.drop_duplicates().shape[0]
check('800 lignes distinctes dans les données brutes', n_uniques == 800, f'{n_uniques}')
check('fichier 800 : 800 lignes', len(d800) == 800, f'{len(d800)}')
check('modélisation + anomalies = 800', len(mod) + len(anom) == 800, f'{len(mod)}+{len(anom)}')
check('anomalie_evidente (fichier 800) = lignes du fichier anomalies',
      d800['anomalie_evidente'].sum() == len(anom))
print(f'    modélisation : {len(mod)} lignes | anomalies : {len(anom)} lignes')

print('\n--- 2. Doublons et identifiants ---')
for nom, d in [('modélisation', mod), ('800', d800), ('anomalies', anom)]:
    check(f'{nom} : reservation_id unique', d['reservation_id'].is_unique)
    check(f'{nom} : aucune ligne dupliquée', not d.duplicated().any())
check('aucune réservation à la fois en modélisation et en anomalies',
      set(mod.reservation_id).isdisjoint(anom.reservation_id))
check('ids du fichier 800 = ids bruts dédoublonnés',
      set(d800.reservation_id) == set(reserv.reservation_id))

print('\n--- 3. Valeurs manquantes ---')
for nom, d in [('modélisation', mod), ('800', d800)]:
    na = d.isna().sum()
    check(f'{nom} : aucune valeur manquante', na.sum() == 0, na[na > 0].to_dict())
check('petit_dejeuner : seulement oui / non / inconnu',
      set(mod.petit_dejeuner) <= {'oui', 'non', 'inconnu'})
n_imputes_attendus = reserv.drop_duplicates()['nb_personnes'].isna().sum()
check('nb_personnes_impute cohérent avec les manquants bruts',
      d800['nb_personnes_impute'].sum() == n_imputes_attendus,
      f"{d800['nb_personnes_impute'].sum()} vs {n_imputes_attendus}")

print('\n--- 4. Modalités et plages de valeurs ---')
check('weekend / a_annule : oui ou non',
      set(mod.weekend) <= {'oui', 'non'} and set(mod.a_annule) <= {'oui', 'non'})
check('canal : mêmes modalités que le brut', set(mod.canal) <= set(reserv.canal.dropna()))
check('segment_declare : affaires / loisir / groupe',
      set(mod.segment_declare) <= {'affaires', 'loisir', 'groupe'})
check('mois_sejour entre 1 et 12', mod.mois_sejour.between(1, 12).all())
check('délai >= 0', (mod.delai_reservation_j >= 0).all())
check('nb_personnes >= 1', (mod.nb_personnes >= 1).all())
check(f'nb_nuits entre 1 et {SEUIL_NUITS - 1} (modélisation)',
      mod.nb_nuits.between(1, SEUIL_NUITS - 1).all())
check(f'prix_nuit_eur > {SEUIL_PRIX} (modélisation)', (mod.prix_nuit_eur > SEUIL_PRIX).all())
check('aucun nb_nuits = 365 ni prix = 0 dans la modélisation',
      not ((mod.nb_nuits == 365) | (mod.prix_nuit_eur == 0)).any())
check('anomalies : toutes dépassent un seuil',
      ((anom.nb_nuits >= SEUIL_NUITS) | (anom.prix_nuit_eur <= SEUIL_PRIX)).all())
check('anomalies : colonne motif renseignée', anom['motif'].notna().all())

print('\n--- 5. Jointure hôtels ---')
cols_hotels = [c for c in hotels.columns if c != 'hotel_id']
check('colonnes hôtels présentes', all(c in mod.columns for c in cols_hotels),
      [c for c in cols_hotels if c not in mod.columns])
check('tous les hotel_id existent dans la table hôtels', set(mod.hotel_id) <= set(hotels.hotel_id))
check('categorie : 3 / 4 / 5 étoiles', set(mod.categorie) <= {'3 étoiles', '4 étoiles', '5 étoiles'})
ref = hotels.set_index('hotel_id')
ok_join = all((mod[c].values == ref.loc[mod.hotel_id, c].values).all() for c in cols_hotels)
check('valeurs hôtels identiques à la table source (pas de décalage de jointure)', ok_join)
check('etoiles cohérent avec categorie',
      (mod.etoiles == mod.categorie.str.extract(r'(\d)')[0].astype(int)).all())

print('\n--- 6. Variables créées ---')
check('a_annule_bin = (a_annule == oui)', (mod.a_annule_bin == (mod.a_annule == 'oui').astype(int)).all())
check('weekend_bin = (weekend == oui)', (mod.weekend_bin == (mod.weekend == 'oui').astype(int)).all())
check('variables binaires : seulement 0 et 1',
      all(set(mod[c]) <= {0, 1} for c in ['a_annule_bin', 'weekend_bin', 'petit_dej_oui', 'petit_dej_inconnu']))
check('petit_dej_oui / petit_dej_inconnu cohérents',
      (mod.petit_dej_oui == (mod.petit_dejeuner == 'oui')).all()
      and (mod.petit_dej_inconnu == (mod.petit_dejeuner == 'inconnu')).all())
check('prix_sejour_eur = prix_nuit_eur × nb_nuits',
      np.allclose(mod.prix_sejour_eur, mod.prix_nuit_eur * mod.nb_nuits))
check('cumul_incoherent cohérent',
      (mod.cumul_incoherent == (mod.nuitees_cumulees_client < mod.nb_nuits)).all())
bins = pd.cut(mod.delai_reservation_j, [-1, 7, 30, 60, 120, np.inf],
              labels=['0-7 j', '8-30 j', '31-60 j', '61-120 j', '>120 j']).astype(str)
check('delai_tranche cohérent avec delai_reservation_j', (mod.delai_tranche == bins).all())
check('saison renseignée (4 valeurs)', set(mod.saison) <= {'hiver', 'printemps', 'ete', 'automne'})

print('\n--- 7. Contrôles de bon sens (informatifs) ---')
taux = mod.a_annule_bin.mean()
print(f"    taux d'annulation (modélisation) : {taux:.1%} (repère du schéma : environ 26 %)")
if not 0.15 <= taux <= 0.40:
    print("    /!\\ AVERTISSEMENT : taux d'annulation hors de 15-40 %, vérifier le fichier brut et a_annule")
print(f'    lignes à cumul incohérent : {mod.cumul_incoherent.sum()} (repère : environ 78 sur 800)')
par_hotel = mod.groupby('hotel_id').size()
print(f'    réservations par hôtel (min / max) : {par_hotel.min()} / {par_hotel.max()} (repère : 14 à 39)')

print('\n=== RÉSULTAT :', f'{sum(resultats)}/{len(resultats)} contrôles OK ===')
assert all(resultats), 'Des contrôles ont échoué : corriger le nettoyage avant de continuer.'
print('Fichiers nettoyés et cohérents.')

--- 1. Comptes de lignes ---
[OK   ] 800 lignes distinctes dans les données brutes
[OK   ] fichier 800 : 800 lignes
[OK   ] modélisation + anomalies = 800
[OK   ] anomalie_evidente (fichier 800) = lignes du fichier anomalies
    modélisation : 794 lignes | anomalies : 6 lignes

--- 2. Doublons et identifiants ---
[OK   ] modélisation : reservation_id unique
[OK   ] modélisation : aucune ligne dupliquée
[OK   ] 800 : reservation_id unique
[OK   ] 800 : aucune ligne dupliquée
[OK   ] anomalies : reservation_id unique
[OK   ] anomalies : aucune ligne dupliquée
[OK   ] aucune réservation à la fois en modélisation et en anomalies
[OK   ] ids du fichier 800 = ids bruts dédoublonnés

--- 3. Valeurs manquantes ---
[OK   ] modélisation : aucune valeur manquante
[OK   ] 800 : aucune valeur manquante
[OK   ] petit_dejeuner : seulement oui / non / inconnu
[OK   ] nb_personnes_impute cohérent avec les manquants bruts

--- 4. Modalités et plages de valeurs ---
[OK   ] weekend / a_annule : oui ou non

**À écrire dans le README (section nettoyage) :** le nombre de contrôles réussis affiché ci-dessus, le nombre d'anomalies isolées et le seuil retenu, le nombre de lignes imputées pour `nb_personnes`, et ce que les vérifications de modalités et d'unités (section 3) ont donné, même si rien d'anormal n'est apparu.

## Quel fichier pour quelle question ?
- **Q1 à Q8** : `reservations_modelisation_794.csv`
- **Q9** : `reservations_propres_800.csv` (contient les anomalies, colonne `anomalie_evidente`) + `anomalies_a_part.csv`. Compter les anomalies avant de les écarter, et écrire l'hypothèse de période (les données n'ont pas d'année) avant de donner des dossiers par mois.
- **Q6** : 14 à 39 réservations par hôtel seulement, donc regrouper par `categorie` ou donner un intervalle de confiance.
- **Q5** : tester avec et sans `nuitees_cumulees_client`.